In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('.')))

from sqlalchemy import text
from db.database import engine
from etl.constants import EUROPEAN_COUNTRIES, DECOUPLING_START_YEAR, DECOUPLING_END_YEAR

In [4]:
query = """
    SELECT 
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_total,
        e.co2_per_capita,
        e.co2_per_gdp,
        e.consumption_co2,
        e.consumption_co2_per_capita,
        e.consumption_co2_per_gdp,
        e.trade_co2,
        e.trade_co2_share,
        e.gdp,
        e.population,
        e.coal_co2,
        e.gas_co2,
        e.oil_co2,
        e.temperature_change_from_co2
    FROM emissions e
    JOIN countries c ON c.id = e.country_id
    WHERE e.year BETWEEN :start AND :end
    ORDER BY c.iso_code, e.year
"""

with engine.connect() as conn:
    df = pd.read_sql(
        text(query),
        conn,
        params={
            'start': DECOUPLING_START_YEAR,
            'end': DECOUPLING_END_YEAR
        }
    )

df_eu = df[df['iso_code'].isin(EUROPEAN_COUNTRIES)].copy()

print(f"Global dataset: {len(df)} rows, {df['iso_code'].nunique()} countries")
print(f"European dataset: {len(df_eu)} rows, {df_eu['iso_code'].nunique()} countries")

2026-06-06 23:23:02,138 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-06-06 23:23:02,144 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-06 23:23:02,147 INFO sqlalchemy.engine.Engine select current_schema()
2026-06-06 23:23:02,147 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-06 23:23:02,152 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-06-06 23:23:02,153 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-06 23:23:02,161 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-06 23:23:02,161 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname

In [5]:
def build_index(df: pd.DataFrame, base_year: int = 1990) -> pd.DataFrame:
    """
    Normalize GDP and CO2 to base_year = 100.
    Allows direct comparison across countries of different sizes.
    """
    df = df.copy()
    
    results = []
    for _, group in df.groupby('iso_code'):
        base = group[group['year'] == base_year]
        if base.empty:
            continue
            
        base_gdp = base['gdp'].values[0]
        base_co2 = base['co2_total'].values[0]
        base_consumption = base['consumption_co2'].values[0]
        
        # Skip if base year values are missing
        if pd.isna(base_gdp) or pd.isna(base_co2):
            continue
            
        group = group.copy()
        group['gdp_index']              = group['gdp'] / base_gdp * 100
        group['co2_index']              = group['co2_total'] / base_co2 * 100
        
        if not pd.isna(base_consumption):
            group['consumption_co2_index'] = group['consumption_co2'] / base_consumption * 100
        else:
            group['consumption_co2_index'] = np.nan
            
        results.append(group)
    
    return pd.concat(results, ignore_index=True)

df_indexed = build_index(df)
df_eu_indexed = build_index(df_eu)

print(f"Indexed dataset: {len(df_eu_indexed)} rows")
print(f"Sample — Poland 1995, 2000, 2005, 2010, 2015, 2020:")
print(df_eu_indexed[df_eu_indexed['iso_code'] == 'POL'][
    ['year', 'gdp_index', 'co2_index', 'consumption_co2_index']
].query('year in [1995, 2000, 2005, 2010, 2015, 2020]').to_string())

Indexed dataset: 1156 rows
Sample — Poland 1995, 2000, 2005, 2010, 2015, 2020:
     year   gdp_index  co2_index  consumption_co2_index
923  1995  116.178057  96.317445              99.898607
928  2000  157.145110  84.313204              90.634862
933  2005  191.828134  85.713792              91.070020
938  2010  256.095460  88.756503             101.600715
943  2015  301.295355  83.049195              92.169933
948  2020  353.613200  80.273262              88.802083


In [9]:
# Which EU countries show GDP growing AND territorial CO2 falling?

latest = df_eu_indexed[df_eu_indexed['year'] == 2022].copy()
latest = latest.dropna(subset=['gdp_index', 'co2_index'])

# Classify countries
def classify_decoupling(row):
    gdp_grew    = row['gdp_index'] > 110     # GDP grew more than 10%
    co2_fell    = row['co2_index'] < 95      # CO2 fell more than 5%
    co2_stable  = row['co2_index'] < 110     # CO2 roughly stable
    
    if gdp_grew and co2_fell:
        return 'Strong decoupling'
    elif gdp_grew and co2_stable:
        return 'Weak decoupling'
    elif gdp_grew:
        return 'No decoupling'
    else:
        return 'Economy shrank'

latest['decoupling_type'] = latest.apply(classify_decoupling, axis=1)

# Visualize
fig = px.scatter(
    latest,
    x='gdp_index',
    y='co2_index',
    color='decoupling_type',
    text='iso_code',
    title='Territorial Decoupling — European Countries (1990=100, measured at 2022)',
    labels={
        'gdp_index': 'GDP Index (1990 = 100)',
        'co2_index': 'CO2 Index (1990 = 100)'
    },
    color_discrete_map={
        'Strong decoupling': 'green',
        'Weak decoupling':   'orange',
        'No decoupling':     'red',
        'Economy shrank':    'gray'
    },
    height=600
)

# Reference lines
fig.add_hline(y=100, line_dash='dash', line_color='gray', annotation_text='CO2 baseline (1990)')
fig.add_vline(x=100, line_dash='dash', line_color='gray', annotation_text='GDP baseline (1990)')

# Green quadrant annotation
fig.add_annotation(
    x=260, y=28,
    text="Green quadrant<br>GDP up, CO2 down",
    showarrow=False,
    font=dict(color='green', size=12)
)

fig.update_traces(textposition='top center')
fig.show()

print("\nDecoupling classification:")
print(latest.groupby('decoupling_type')['country'].apply(list))


Decoupling classification:
decoupling_type
Economy shrank                                      [Georgia, Ukraine]
No decoupling                                [Cyprus, Ireland, Norway]
Strong decoupling    [Albania, Belgium, Bulgaria, Belarus, Switzerl...
Weak decoupling                                       [Austria, Spain]
Name: country, dtype: object


In [12]:
# Pick top 8 decouplers + Poland for context
top_decouplers = latest[latest['decoupling_type'] == 'Strong decoupling']['iso_code'].tolist()

# Always include Poland for context
focus_countries = top_decouplers[:8] + ['POL']
focus_countries = list(set(focus_countries))  # deduplicate

df_focus = df_eu_indexed[df_eu_indexed['iso_code'].isin(focus_countries)]

fig = px.line(
    df_focus,
    x='year',
    y='co2_index',
    color='iso_code',
    title='CO2 Trajectory (1990 = 100) - Top Decouplers',
    labels={'co2_index': 'CO2 Index (1990 = 100)', 'year': 'Year'},
    height=500
)
fig.add_hline(y=100, line_dash='dash', line_color='gray')
fig.show()

# Same for GDP
fig2 = px.line(
    df_focus,
    x='year',
    y='gdp_index',
    color='iso_code',
    title='GDP Trajectory (1990 = 100) - Top Decouplers',
    labels={'gdp_index': 'GDP Index (1990 = 100)', 'year': 'Year'},
    height=500
)
fig2.add_hline(y=100, line_dash='dash', line_color='gray')
fig2.show()

In [21]:
# TERRITORIAL vs CONSUMPTION CO2
# Countries where consumption stayed high = fake decoupling

latest_full = df_eu_indexed[
    (df_eu_indexed['year'] == 2022) &
    (df_eu_indexed['consumption_co2_index'].notna())
].copy()

latest_full['gap'] = latest_full['co2_index'] - latest_full['consumption_co2_index']
# Positive gap = territorial looks better than reality (imported emissions)
# Negative gap = territorial looks worse than reality (exported emissions)

latest_full = latest_full.sort_values('gap', ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Territorial<br>CO2 index',
    x=latest_full['iso_code'],
    y=latest_full['co2_index'],
    marker_color='steelblue'
))

fig.add_trace(go.Bar(
    name='Consumption<br>CO2 index',
    x=latest_full['iso_code'],
    y=latest_full['consumption_co2_index'],
    marker_color='coral'
))

fig.add_hline(y=100, line_dash='dash', line_color='gray')
fig.update_layout(
    title='Territorial vs Consumption CO2 - 2022<br><sup>Gap = difference between what countries emit vs what they consume</sup>',
    barmode='group',
    height=600,
    xaxis_tickangle=-45
)
fig.show()

print("\nBiggest 'fake decouplers' (territorial looks better than reality):")
print(latest_full[latest_full['gap'] > 2][['country', 'co2_index', 'consumption_co2_index', 'gap']].to_string())

print("\nCountries that look worse than reality (exporters):")
print(latest_full[latest_full['gap'] < -30][['country', 'co2_index', 'consumption_co2_index', 'gap']].to_string())


Biggest 'fake decouplers' (territorial looks better than reality):
      country  co2_index  consumption_co2_index       gap
1120   Sweden  63.294843              59.142009  4.152834
440   Finland  63.837599              59.534690  4.302910
66    Austria  98.814941              93.802594  5.012347

Countries that look worse than reality (exporters):
         country  co2_index  consumption_co2_index         gap
848        Malta  73.094355             493.993258 -420.898903
100      Belgium  73.889558             172.101217  -98.211659
542      Georgia  80.808081             164.234857  -83.426776
202  Switzerland  74.635892             144.851712  -70.215820
168      Belarus  52.665984             104.387680  -51.721696
610      Croatia  76.840725             118.812226  -41.971501
814       Latvia  33.607975              68.117394  -34.509420
338      Denmark  54.404697              86.629132  -32.224435


In [ ]:
# trade_co2 positive = net importer of emissions
# trade_co2 negative = net exporter of emissions

df_trade = df_eu[
    (df_eu['year'] >= 1990) &
    (df_eu['trade_co2'].notna())
].copy()

# Latest year available
latest_trade = df_trade[df_trade['year'] == df_trade['year'].max()].copy()
latest_trade = latest_trade.sort_values('trade_co2', ascending=True)

colors = ['green' if x < 0 else 'red' for x in latest_trade['trade_co2']]

fig = go.Figure(go.Bar(
    x=latest_trade['iso_code'],
    y=latest_trade['trade_co2'],
    marker_color=colors,
    text=latest_trade['country'],
))

fig.add_hline(y=0, line_color='black')
fig.update_layout(
    title='Net Trade CO2 - European Countries<br><sup>Negative = exporting emissions | Positive = importing emissions</sup>',
    yaxis_title='Net CO2 from trade (million tonnes)',
    height=500,
    xaxis_tickangle=-45
)
fig.show()

# Trade CO2 as % of total - who is most affected?
latest_trade['trade_impact'] = (latest_trade['trade_co2'] / latest_trade['co2_total'] * 100).round(1)

print("\nTrade CO2 as % of territorial emissions:")
print(latest_trade[['country', 'trade_co2', 'trade_co2_share', 'trade_impact']]
      .sort_values('trade_impact').to_string())


Trade CO2 as % of territorial emissions:
             country  trade_co2  trade_co2_share  trade_impact
5303          Poland     -9.388           -3.314          -3.3
4929          Norway      0.101            0.260           0.3
713         Bulgaria      0.809            2.345           2.3
3943      Luxembourg      0.491            7.181           7.2
849          Belarus      6.812           12.222          12.2
6153        Slovakia      4.269           13.875          13.9
5541         Romania     10.385           15.303          15.3
6901         Ukraine     26.980           19.365          19.4
1733          Cyprus      1.436           19.982          20.0
1767         Czechia     16.793           20.176          20.2
2107           Spain     45.155           20.957          21.0
3059         Ireland      7.266           21.645          21.6
169          Albania      0.965           21.838          21.8
5371        Portugal      9.838           26.149          26.1
4895     Neth